In [1]:
import sys; sys.path.insert(0, '..')
from shared.training_pipeline import init_ee

init_ee()


/Users/ashutoshsaxena/Desktop/ml-central/Geo-Engine-Classifier/.venv/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/ashutoshsaxena/Desktop/ml-central/Geo-Engine-Classifier/.venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/ashutoshsaxena/Desktop/ml-central/Geo-Engine-Classifier/.venv/lib/python3.9/site-packages/google/api_core/_py

Earth Engine initialized successfully.


In [2]:
import ee
from shared.training_pipeline import get_campus_geometry

campus = get_campus_geometry()


In [3]:
area = campus.area()
print("Campus Area (m²):", area.getInfo())
print("Campus Area (km²):", area.divide(1e6).getInfo())

Campus Area (m²): 1995406.3843645903
Campus Area (km²): 1.9954063843647254


In [4]:
import math

targeted_area_sqm = float(area.getInfo())*2

point = {
    "type": "Point",
    "coordinates": [79.88361463362526,23.142324404777362]
}


centre = ee.Geometry(point)


circle_radius = math.sqrt(targeted_area_sqm / math.pi)

#region of interest
roi = centre.buffer(circle_radius)

In [5]:
from shared.training_pipeline import build_image, BANDS

# Build test area image using shared pipeline
new_image = build_image(roi, '2026-01-01', '2026-02-28', cloud_cover=5)
bands = BANDS


In [6]:
import geemap
Map = geemap.Map()

Map.centerObject(roi, 16)

Map.addLayer(new_image, {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 0.3,
}, 'Test Area')

Map

Map(center=[23.142325561034358, 79.8836146684978], controls=(WidgetControl(options=['position', 'transparent_b…

In [7]:
# NDVI and bands already set by build_image() above
# (ndvi band included in new_image, bands imported from shared module)


In [8]:
# Train XGB classifier using shared pipeline
# (eliminates redundant re-training code)
from shared.training_pipeline import (
    train_model, load_training_points, BANDS
)

classifier, train_set, test_set = train_model("xgb")
bands = BANDS

# Print training/test info
forest_points, non_forest_points, _ = load_training_points()
print('Forest points:', forest_points.size().getInfo())
print('Non-forest points:', non_forest_points.size().getInfo())

# Evaluate on train/test split
validated = test_set.classify(classifier)
confusion_matrix = validated.errorMatrix(
    actual='label',
    predicted='classification'
)
print(confusion_matrix.getInfo())
print('Accuracy:', confusion_matrix.accuracy().getInfo())
print('Kappa:', confusion_matrix.kappa().getInfo())


Forest points: 250
Non-forest points: 250
[[81, 4], [4, 57]]
Accuracy: 0.9452054794520548
Kappa: 0.8873674059787849


In [9]:
new_classified = new_image.select(bands).classify(classifier)
new_smooth_classification = new_classified.focalMode(1)

In [10]:
Map = geemap.Map()
Map.centerObject(roi, 16)

Map.addLayer(new_image, {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 0.3
}, 'Test Area RGB')

Map.addLayer(new_smooth_classification, {
    'min': 0, 
    'max': 1, 
    'palette': ['lightgray', 'darkgreen']
}, 'Test Area Forest Map')

Map

Map(center=[23.142325561034358, 79.8836146684978], controls=(WidgetControl(options=['position', 'transparent_b…

In [11]:
test_forest_points = ee.FeatureCollection('users/ashutoshsaxena703/forestnew')
test_non_forest_points = ee.FeatureCollection('users/ashutoshsaxena703/nonforestnew')

test_points = test_forest_points.merge(test_non_forest_points)

In [12]:
# Count number of features in the forest and non-forest collections
def _count(fc):
    try:
        return int(fc.size().getInfo())
    except Exception as e:
        print("Error counting collection:", e)
        return None

test_forest_count = _count(test_forest_points) if 'forest_points' in locals() else None
test_non_forest_count = _count(test_non_forest_points) if 'non_forest_points' in locals() else None

print("Forest points:", test_forest_count)
print("Non-forest points:", test_non_forest_count)

Forest points: 240
Non-forest points: 240


In [13]:
validation_data = new_image.select(bands).sampleRegions(
    collection=test_points,
    properties=['label'],
    scale=10
)

validation_data = validation_data.filter(ee.Filter.notNull(bands + ['label']))

validated = validation_data.classify(classifier)

In [14]:

confusion_matrix = validated.errorMatrix(
    actual='label',
    predicted='classification'
)

print("Confusion Matrix: ", confusion_matrix.getInfo())
print("Test Area Accuracy: ", confusion_matrix.accuracy().getInfo())
print("Test Area Kappa: ", confusion_matrix.kappa().getInfo())

Confusion Matrix:  [[219, 21], [30, 210]]
Test Area Accuracy:  0.89375
Test Area Kappa:  0.7875000000000001
